# 🧬 Elite Training Pipeline — Glucose Forecasting

**Architecture:**  
1. **Elite Selection** → Top 100 patients by data volume  
2. **Cleaning & Resampling** → Strict 5-min grid, interpolation  
3. **Bio-Brain Features** → IOB, COB, Velocity, Acceleration, Circadian  
4. **Target & Polish** → 30-min forecast target, correlation analysis  

**RAM Safety:** Chunked per-patient processing via `scan_parquet`.  

---

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 0: Setup
# ══════════════════════════════════════════════════════════════════════
!pip install polars plotly pyarrow -q

import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import numpy as np
import math
import os
import gc
import time

from google.colab import drive
drive.mount('/content/drive')

print(f"Polars : {pl.__version__}")
print(f"Run at : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 0b: Configuration
# ══════════════════════════════════════════════════════════════════════
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data"

RAW_TRAIN   = f"{BASE_PATH}/raw/metabonet_public_train.parquet"
OUTPUT_FILE = f"{BASE_PATH}/processed/train_elite_physio_engineered.parquet"

# Pipeline parameters
N_ELITE       = 100       # Top N patients to keep
RESAMPLE_FREQ = "5m"      # Target grid frequency
INTERP_LIMIT  = 3         # Max consecutive NULLs to interpolate (3 × 5min = 15min)
FORECAST_STEPS = 6        # Target shift (6 × 5min = 30 min)
BATCH_SIZE     = 5        # Patients per processing batch

# Ensure output directory exists
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

print(f"📁 Input : {RAW_TRAIN}  — exists: {os.path.exists(RAW_TRAIN)}")
print(f"📁 Output: {OUTPUT_FILE}")
print(f"\n⚙️  Config:")
print(f"   Elite patients : {N_ELITE}")
print(f"   Grid frequency : {RESAMPLE_FREQ}")
print(f"   Interp limit   : {INTERP_LIMIT} steps (={INTERP_LIMIT*5} min)")
print(f"   Forecast horizon: {FORECAST_STEPS} steps (={FORECAST_STEPS*5} min)")
print(f"   Batch size      : {BATCH_SIZE} patients")

---
## STEP 1 — Elite Patient Selection 🏆
Scan lazy → group → pick Top 100 → discard the rest immediately.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# STEP 1: Elite Patient Selection
# ══════════════════════════════════════════════════════════════════════
t0 = time.time()

q = pl.scan_parquet(RAW_TRAIN)
schema = q.collect_schema()
print(f"Schema: {len(schema.names())} columns")
for c in schema.names():
    print(f"  {c:30s} → {schema[c]}")

# Count rows per patient (lazy — only the aggregation result hits RAM)
patient_counts = (
    q.group_by("id")
    .agg(pl.len().alias("row_count"))
    .sort("row_count", descending=True)
    .collect()
)

total_patients = len(patient_counts)
total_rows = patient_counts["row_count"].sum()

print(f"\n📊 Full dataset: {total_rows:,} rows across {total_patients} patients")

# Select Top N
elite = patient_counts.head(N_ELITE)
elite_ids = elite["id"].to_list()
elite_rows = elite["row_count"].sum()

print(f"\n⭐ Elite {N_ELITE} patients:")
print(f"   Total rows: {elite_rows:,} ({elite_rows/total_rows*100:.1f}% of dataset)")
print(f"   Min rows:   {elite['row_count'].min():,}")
print(f"   Max rows:   {elite['row_count'].max():,}")
print(f"   Avg rows:   {elite['row_count'].mean():,.0f}")

print(f"\n⏱️  Step 1 completed in {time.time()-t0:.1f}s")

In [ ]:
# ── Quick visual of elite patients ───────────────────────────────────
fig = px.bar(
    elite.to_pandas(), x="id", y="row_count",
    title=f"🏆 Elite {N_ELITE} Patients — Data Volume",
    labels={"id": "Patient ID", "row_count": "Rows"},
    color="row_count", color_continuous_scale="Viridis",
)
fig.update_layout(template="plotly_dark", height=450, xaxis_tickangle=-90,
                  xaxis=dict(dtick=1))
fig.show()

---
## STEP 2 + 3 + 4 — Per-Patient Processing Engine 🔧
Each patient is loaded, cleaned, resampled, feature-engineered, and targeted **individually**.  
Results are appended to a list and concatenated at the end.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CORE FUNCTIONS
# ══════════════════════════════════════════════════════════════════════

# Columns to select from raw data
RAW_COLS = ["id", "date", "CGM", "basal", "bolus", "carbs"]
# Add 'insulin' if it exists in schema, otherwise we'll derive it from bolus
if "insulin" in schema.names():
    RAW_COLS.append("insulin")
    HAS_INSULIN = True
else:
    HAS_INSULIN = False
    print("⚠️  'insulin' column not found. Will use 'bolus' as insulin proxy.")

# Filter to only columns that exist
RAW_COLS = [c for c in RAW_COLS if c in schema.names()]
print(f"Selecting columns: {RAW_COLS}")


def ensure_datetime(df: pl.DataFrame) -> pl.DataFrame:
    """Convert 'date' to Datetime if it isn't already."""
    dt = df["date"].dtype
    if dt == pl.Datetime or str(dt).startswith("Datetime"):
        return df.rename({"date": "datetime"})
    elif dt in [pl.Int64, pl.UInt64]:
        # Try nanoseconds
        df = df.with_columns(pl.from_epoch(pl.col("date"), time_unit="ns").alias("datetime"))
        year = df["datetime"][0].year
        if year < 2000 or year > 2030:
            df = df.with_columns(pl.from_epoch(pl.col("date"), time_unit="us").alias("datetime"))
        return df.drop("date")
    else:
        return df.rename({"date": "datetime"})


def resample_patient(df: pl.DataFrame) -> pl.DataFrame:
    """
    STEP 2: Resample a single patient onto a strict 5-min grid.
    - CGM: linear interpolation (limit=3 → 15min)
    - Insulin/Carbs: fill null → 0
    - Basal: forward fill
    """
    pid = df["id"][0]

    # Sort by time
    df = df.sort("datetime")

    # Create the 5-min grid spanning this patient's full range
    t_min = df["datetime"].min()
    t_max = df["datetime"].max()

    # Snap to nearest 5-min boundary
    t_min_snapped = t_min.replace(second=0, microsecond=0)
    # Round minute down to nearest 5
    t_min_snapped = t_min_snapped.replace(minute=(t_min_snapped.minute // 5) * 5)

    grid = pl.DataFrame({
        "datetime": pl.datetime_range(
            t_min_snapped, t_max, interval="5m", eager=True
        )
    })

    # Remove duplicates (keep last if multiple readings in same slot)
    df = df.unique(subset=["datetime"], keep="last")

    # Join onto grid
    merged = grid.join(df.drop("id"), on="datetime", how="left")

    # ── Interpolation Strategy ────────────────────────────────────────
    # CGM: linear interpolation with limit
    merged = merged.with_columns(
        pl.col("CGM").interpolate().alias("CGM_interp")
    )

    # Enforce interpolation limit: only fill gaps ≤ INTERP_LIMIT consecutive NULLs
    # Create a mask: 1 where original was null, 0 otherwise
    merged = merged.with_columns(
        pl.col("CGM").is_null().cast(pl.Int32).alias("_null_flag")
    )
    # Count consecutive nulls using cumsum trick
    merged = merged.with_columns(
        (pl.col("_null_flag") * (
            pl.col("_null_flag")
            .cum_sum()
            - (pl.col("_null_flag") == 0).cast(pl.Int32)
              .cum_sum()
              .forward_fill()
              .fill_null(0)
              .gather(
                  (pl.col("_null_flag") == 0)
                  .arg_true()
              )
              .gather(pl.arange(0, merged.height).alias("idx"))
        )).alias("_consec")
    )
    # This is complex — simpler approach: use group-based consecutive null counting
    # Let's use a simpler method: mark runs of nulls by their length
    # and only keep interpolated values where run length <= INTERP_LIMIT

    # Simplified: mark null runs
    is_null = merged["CGM"].is_null().to_list()
    run_lengths = []
    count = 0
    for v in is_null:
        if v:
            count += 1
            run_lengths.append(count)
        else:
            count = 0
            run_lengths.append(0)

    # Backfill run lengths so we know the full run length at each position
    max_runs = run_lengths.copy()
    current_max = 0
    for i in range(len(max_runs) - 1, -1, -1):
        if max_runs[i] > 0:
            if i + 1 < len(max_runs) and max_runs[i + 1] > max_runs[i]:
                current_max = max_runs[i + 1]
            else:
                current_max = max_runs[i]
            max_runs[i] = current_max
        else:
            current_max = 0

    # Apply limit: keep interpolation only where gap <= INTERP_LIMIT
    cgm_final = []
    interp_vals = merged["CGM_interp"].to_list()
    orig_vals = merged["CGM"].to_list()
    for i in range(len(orig_vals)):
        if orig_vals[i] is not None:
            cgm_final.append(orig_vals[i])
        elif max_runs[i] <= INTERP_LIMIT:
            cgm_final.append(interp_vals[i])
        else:
            cgm_final.append(None)

    merged = merged.with_columns(
        pl.Series("CGM", cgm_final, dtype=pl.Float64)
    ).drop(["CGM_interp", "_null_flag"])

    # Drop the _consec column if it exists
    if "_consec" in merged.columns:
        merged = merged.drop("_consec")

    # ── Fill strategies for other columns ─────────────────────────────
    fill_zero_cols = [c for c in ["bolus", "carbs"] if c in merged.columns]
    if HAS_INSULIN and "insulin" in merged.columns:
        fill_zero_cols.append("insulin")

    for c in fill_zero_cols:
        merged = merged.with_columns(pl.col(c).fill_null(0.0).cast(pl.Float64))

    if "basal" in merged.columns:
        merged = merged.with_columns(
            pl.col("basal").forward_fill().fill_null(0.0).cast(pl.Float64)
        )

    # Add patient id back
    merged = merged.with_columns(pl.lit(pid).alias("id"))

    # Cast CGM
    merged = merged.with_columns(pl.col("CGM").cast(pl.Float64))

    return merged


def engineer_features(df: pl.DataFrame) -> pl.DataFrame:
    """
    STEP 3: Advanced feature engineering (Bio-Brain).
    Assumes data is already sorted by datetime and on a 5-min grid.
    """
    # Insulin column: use 'insulin' if available, else 'bolus'
    ins_col = "insulin" if "insulin" in df.columns else "bolus"

    df = df.with_columns([
        # ── A. Biological Simulation (IOB & COB) ──────────────────────
        pl.col(ins_col).ewm_mean(span=48).alias("IOB"),       # ~4h decay
        pl.col("carbs").ewm_mean(span=36).alias("COB"),        # ~3h absorption

        # ── B. Kinetic Features ───────────────────────────────────────
        (pl.col("CGM") - pl.col("CGM").shift(1)).alias("Vel_5min"),
        (pl.col("CGM") - pl.col("CGM").shift(3)).alias("Vel_15min"),

        # ── C. Rolling Statistics (60-min window = 12 steps) ──────────
        pl.col("CGM").rolling_mean(window_size=12).alias("Roll_Mean_60"),
        pl.col("CGM").rolling_std(window_size=12).alias("Roll_Std_60"),
    ])

    # Acceleration (depends on Vel_15min)
    df = df.with_columns(
        (pl.col("Vel_15min") - pl.col("Vel_15min").shift(3)).alias("Acc_15min")
    )

    # ── C. Circadian Rhythm ───────────────────────────────────────────
    df = df.with_columns([
        pl.col("datetime").dt.hour().alias("_hour"),
    ])
    df = df.with_columns([
        (2 * math.pi * pl.col("_hour") / 24.0).sin().alias("Hour_Sin"),
        (2 * math.pi * pl.col("_hour") / 24.0).cos().alias("Hour_Cos"),
    ]).drop("_hour")

    # ── Interaction Feature ────────────────────────────────────────────
    df = df.with_columns(
        (pl.col("IOB") * pl.col("Vel_15min")).alias("Insulin_Sensitivity_Risk")
    )

    return df


def create_target(df: pl.DataFrame) -> pl.DataFrame:
    """
    STEP 4: Create forecast target and sanitize.
    """
    df = df.with_columns(
        pl.col("CGM").shift(-FORECAST_STEPS).alias("Target_CGM")
    )

    # Drop rows where target or key features are null
    critical_cols = [
        "CGM", "Target_CGM", "IOB", "COB",
        "Vel_5min", "Vel_15min", "Acc_15min",
        "Roll_Mean_60", "Roll_Std_60",
    ]
    existing = [c for c in critical_cols if c in df.columns]
    df = df.drop_nulls(subset=existing)

    return df


def process_single_patient(patient_id: str, q: pl.LazyFrame) -> pl.DataFrame:
    """
    Full pipeline for a single patient: filter → clean → resample → features → target.
    """
    # Load only this patient + only needed columns
    available_cols = [c for c in RAW_COLS if c in q.collect_schema().names()]
    df = (
        q.filter(pl.col("id") == patient_id)
        .select(available_cols)
        .collect()
    )

    if len(df) < 50:  # Skip patients with too little data
        return None

    # Ensure datetime
    df = ensure_datetime(df)

    # Cast CGM
    df = df.with_columns(pl.col("CGM").cast(pl.Float64, strict=False))

    # Step 2: Resample
    df = resample_patient(df)

    if len(df) < 50:
        return None

    # Step 3: Feature engineering
    df = engineer_features(df)

    # Step 4: Target creation + sanitization
    df = create_target(df)

    return df


print("✅ All processing functions defined.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# MAIN PROCESSING LOOP (Chunked, RAM-Safe)
# ══════════════════════════════════════════════════════════════════════
t_start = time.time()

results = []
errors = []
skipped = []

n_total = len(elite_ids)
print(f"Processing {n_total} elite patients in batches of {BATCH_SIZE}...")
print("=" * 70)

for i, pid in enumerate(elite_ids):
    try:
        df_processed = process_single_patient(pid, q)

        if df_processed is not None and len(df_processed) > 0:
            results.append(df_processed)
            status = f"✅ {len(df_processed):>8,} rows"
        else:
            skipped.append(pid)
            status = "⏭️  skipped (too little data)"

    except Exception as e:
        errors.append((pid, str(e)))
        status = f"❌ ERROR: {str(e)[:50]}"

    # Progress report every BATCH_SIZE patients
    if (i + 1) % BATCH_SIZE == 0 or i == n_total - 1:
        elapsed = time.time() - t_start
        pct = (i + 1) / n_total * 100
        est_total = elapsed / (i + 1) * n_total
        est_remain = est_total - elapsed
        print(
            f"  [{i+1:3d}/{n_total}] {pct:5.1f}% | "
            f"Patient '{pid}' → {status} | "
            f"Elapsed: {elapsed:.0f}s | ETA: {est_remain:.0f}s"
        )

    # Periodic garbage collection
    if (i + 1) % (BATCH_SIZE * 2) == 0:
        gc.collect()

elapsed_total = time.time() - t_start
print("\n" + "=" * 70)
print(f"✅ Processing complete in {elapsed_total:.1f}s ({elapsed_total/60:.1f} min)")
print(f"   Successful: {len(results)}")
print(f"   Skipped:    {len(skipped)}")
print(f"   Errors:     {len(errors)}")

if errors:
    print("\n⚠️  Error details:")
    for pid, err in errors:
        print(f"   Patient '{pid}': {err}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CONCATENATE & SAVE
# ══════════════════════════════════════════════════════════════════════
print("Concatenating all processed patients...")

df_final = pl.concat(results, how="diagonal_relaxed")

# Free individual chunks
del results
gc.collect()

print(f"\n📊 Final Dataset:")
print(f"   Shape   : {df_final.shape}")
print(f"   Patients: {df_final['id'].n_unique()}")
print(f"   RAM     : ~{df_final.estimated_size('mb'):.1f} MB")
print(f"\n   Columns:")
for c, dt in df_final.schema.items():
    n = df_final[c].null_count()
    print(f"     {c:30s} {str(dt):15s} nulls: {n:>8,} ({n/len(df_final)*100:.2f}%)")

In [ ]:
# ── Cast all feature columns to Float32 for training efficiency ──────
float_cols = [
    c for c, dt in df_final.schema.items()
    if dt in [pl.Float64, pl.Float32] and c not in ["datetime"]
]

df_final = df_final.with_columns([
    pl.col(c).cast(pl.Float32) for c in float_cols
])

print(f"Casted {len(float_cols)} numeric columns to Float32.")
print(f"RAM after cast: ~{df_final.estimated_size('mb'):.1f} MB")

# Save
print(f"\n💾 Saving to: {OUTPUT_FILE}")
df_final.write_parquet(OUTPUT_FILE, use_pyarrow=True, compression="zstd")

file_size = os.path.getsize(OUTPUT_FILE) / (1024**2)
print(f"✅ Saved! File size: {file_size:.1f} MB")

---
## STEP 5 — Validation & Correlation Analysis 📈

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# VALIDATION: Rows per patient distribution
# ══════════════════════════════════════════════════════════════════════
final_counts = (
    df_final.group_by("id")
    .agg(pl.len().alias("rows"))
    .sort("rows", descending=True)
)

print(f"Final dataset: {len(df_final):,} rows × {df_final.shape[1]} cols")
print(f"Patients: {final_counts.height}")
print(f"Rows/patient — min: {final_counts['rows'].min():,}  max: {final_counts['rows'].max():,}  avg: {final_counts['rows'].mean():,.0f}")

fig = px.bar(
    final_counts.to_pandas(), x="id", y="rows",
    title="📊 Rows per Patient (Final Engineered Dataset)",
    color="rows", color_continuous_scale="Plasma",
)
fig.update_layout(template="plotly_dark", height=400, xaxis_tickangle=-90)
fig.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CORRELATION ANALYSIS: Features vs Target_CGM
# ══════════════════════════════════════════════════════════════════════
feature_cols = [
    "CGM", "IOB", "COB",
    "Vel_5min", "Vel_15min", "Acc_15min",
    "Roll_Mean_60", "Roll_Std_60",
    "Hour_Sin", "Hour_Cos",
    "Insulin_Sensitivity_Risk",
    "Target_CGM",
]

# Filter to existing columns
feature_cols = [c for c in feature_cols if c in df_final.columns]

# Compute correlation matrix using pandas (small subset, safe for RAM)
corr_df = df_final.select(feature_cols).to_pandas()
corr_matrix = corr_df.corr()

print("Correlation with Target_CGM:")
print("=" * 50)
target_corr = corr_matrix["Target_CGM"].drop("Target_CGM").sort_values(ascending=False)
for feat, val in target_corr.items():
    bar = "█" * int(abs(val) * 30)
    sign = "+" if val > 0 else "-"
    print(f"  {feat:30s} {sign}{abs(val):.4f}  {bar}")

In [ ]:
# ── Correlation Heatmap ──────────────────────────────────────────────
fig = px.imshow(
    corr_matrix.values,
    x=corr_matrix.columns.tolist(),
    y=corr_matrix.index.tolist(),
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="🔬 Feature Correlation Matrix",
    text_auto=".2f",
)
fig.update_layout(
    template="plotly_dark",
    height=700, width=800,
    xaxis_tickangle=-45,
)
fig.show()

In [ ]:
# ── Target vs Feature scatter for top predictors ─────────────────────
top_features = target_corr.head(4).index.tolist()

# Sample for scatter performance
sample_n = min(50_000, len(df_final))
sample = df_final.sample(n=sample_n, seed=42).to_pandas()

fig = make_subplots(rows=2, cols=2, subplot_titles=top_features)

for i, feat in enumerate(top_features):
    row, col = divmod(i, 2)
    fig.add_trace(
        go.Scattergl(
            x=sample[feat], y=sample["Target_CGM"],
            mode="markers", marker=dict(size=2, opacity=0.3, color="#4ECDC4"),
            name=feat,
        ),
        row=row+1, col=col+1
    )

fig.update_layout(
    title="🎯 Top 4 Features vs Target_CGM (50k sample)",
    template="plotly_dark", height=700, showlegend=False,
)
fig.show()

In [ ]:
# ── Sample trace: CGM + IOB + COB for one patient ────────────────────
sample_id = elite_ids[0]
trace = (
    df_final.filter(pl.col("id") == sample_id)
    .sort("datetime")
    .head(2000)  # First ~7 days
    .to_pandas()
)

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    subplot_titles=("CGM + Target", "IOB (Insulin On Board)", "COB (Carbs On Board)"),
    vertical_spacing=0.06,
)

fig.add_trace(go.Scatter(x=trace["datetime"], y=trace["CGM"], name="CGM",
                         line=dict(color="#00D4AA", width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=trace["datetime"], y=trace["Target_CGM"], name="Target (+30m)",
                         line=dict(color="#FF6B6B", width=1, dash="dot")), row=1, col=1)
fig.add_hline(y=70, line_dash="dash", line_color="red", row=1, col=1)
fig.add_hline(y=180, line_dash="dash", line_color="orange", row=1, col=1)

fig.add_trace(go.Scatter(x=trace["datetime"], y=trace["IOB"], name="IOB",
                         line=dict(color="#9B59B6", width=1), fill="tozeroy"), row=2, col=1)

fig.add_trace(go.Scatter(x=trace["datetime"], y=trace["COB"], name="COB",
                         line=dict(color="#F39C12", width=1), fill="tozeroy"), row=3, col=1)

fig.update_layout(
    title=f"🧬 Bio-Brain Features — Patient '{sample_id}' (first ~7 days)",
    template="plotly_dark", height=750,
)
fig.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# FINAL VERDICT
# ══════════════════════════════════════════════════════════════════════

n_final_patients = df_final["id"].n_unique()
n_final_rows = len(df_final)
n_features = len([c for c in df_final.columns if c not in ["id", "datetime", "Target_CGM"]])

print("")
print("╔══════════════════════════════════════════════════════════════════════╗")
print("║              🏁  ELITE TRAINING PIPELINE — COMPLETE               ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print(f"║  Output File  : {os.path.basename(OUTPUT_FILE):51s}  ║")
print(f"║  File Size    : {file_size:>8.1f} MB                                       ║")
print(f"║  Total Rows   : {n_final_rows:>12,}                                    ║")
print(f"║  Patients     : {n_final_patients:>12,}                                    ║")
print(f"║  Features     : {n_features:>12,}                                    ║")
print(f"║  Target       : Target_CGM (CGM +{FORECAST_STEPS*5} min)                        ║")
print("║                                                                      ║")
print("║  🧬 Engineered Features:                                            ║")
print("║     IOB, COB, Vel_5min, Vel_15min, Acc_15min,                       ║")
print("║     Roll_Mean_60, Roll_Std_60, Hour_Sin, Hour_Cos,                  ║")
print("║     Insulin_Sensitivity_Risk                                         ║")
print("║                                                                      ║")
print(f"║  Top Predictor: {target_corr.index[0]:20s} (r = {target_corr.values[0]:+.4f})             ║")
print(f"║  Processing   : {elapsed_total:>8.1f}s ({elapsed_total/60:.1f} min)                          ║")
print("║                                                                      ║")
print("║  ✅ READY FOR XGBOOST / LSTM TRAINING                               ║")
print("╚══════════════════════════════════════════════════════════════════════╝")

# Cleanup
del df_final, corr_df
gc.collect()
print("\n🧹 All DataFrames freed from RAM.")